In [ ]:
# ============================================================
#  Hybrid 1D CNN + RNN Ensemble for Automatic Modulation Classification (AMC)
#
# Purpose
# -------
# This script:
#   - Loads the RadioML 2016.10A dataset (raw I/Q, shape (N,2,128))
#   - Rebuilds the best 1D CNN and best RNN from *your existing ablation runs*
#     - CNN manifests live under: /.../amc_runs/cnn_1d/<run_tag>/*_manifest.json
#     - RNN manifests live under: /.../amc_runs/<rnn_run_tag>/<cfg_name>/metrics/manifest.json
#   - Loads their saved checkpoints
#   - Builds a hybrid ensemble model:
#       logits_hybrid = w_cnn * logits_cnn + w_rnn * logits_rnn
#     (default: w_cnn = w_rnn = 0.5)
#   - Evaluates test accuracy and Accuracy-vs-SNR curves for:
#       - CNN alone
#       - RNN alone
#       - CNN+RNN hybrid
#
# Outputs
# -------
# Under /content/drive/MyDrive/amc_runs/hybrid_<timestamp>/ :
#   - hybrid_manifest.json          (test metrics and paths)
#   - snr_curves_hybrid.png         (SNR vs accuracy for CNN/RNN/hybrid)
#   - test_acc_bar_hybrid.png       (bar chart of test accuracies)
#   - cnn_rnn_best_manifests.json   (the two best model manifests)
#
# Requirements
# ------------
# - You have already run:
#     * 3cnn_codeonly.py  (1D CNN ablations)
#     * 4rnn_codeonly.py  (RNN ablations)
# - They produced manifests + checkpoints on your Drive.
# - RadioML 2016.10A .pkl is located at `data_path` below.
# ============================================================

import os, sys, json, math, time, random, glob, platform
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print(f"Python: {sys.version.split()[0]}  |  Platform: {platform.platform()}")

# ========= Colab Drive + basic paths =========
from google.colab import drive
drive.mount("/content/drive")

# --- YOU: update these three paths if needed ---
data_path    = "/content/drive/MyDrive/AMC_datasets/RadioML2016_10A_dataset.pkl"

# Best CNN run root (from your earlier log, already known):
CNN_RUN_ROOT = "/content/drive/MyDrive/amc_runs/cnn_1d/20251130-044350"

# Best RNN run root (you need to paste the actual RUN_TAG from your RNN script output):
RNN_RUN_ROOT = "/content/drive/MyDrive/amc_runs/amc_rnn_results_20251130-001903"

# Hybrid output folder
HYBRID_TAG = datetime.now().strftime("hybrid_%Y%m%d-%H%M%S")
HYBRID_ROOT = os.path.join("/content/drive/MyDrive/amc_runs", HYBRID_TAG)
os.makedirs(HYBRID_ROOT, exist_ok=True)
print("Hybrid artifacts will go to:", HYBRID_ROOT)

# ========= Helper: global seed =========
def set_global_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_global_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ========= Load RadioML dataset via existing loader from your repo =========
# We reuse your repo's data loader since it's already correct.
REPO_URL = "https://github.com/Koosmaster/Tutorial-Automatic-Modulation-Classification.git"
REPO_DIR = "/content/Tutorial-Automatic-Modulation-Classification"
SRC_DIR  = os.path.join(REPO_DIR, "src")

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 "$REPO_URL" "$REPO_DIR"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from data import load_radioml_pkl_dataset  # your helper

print("Loading RadioML 2016.10A from:", data_path)
radioml_data, mod_classes, snr_values = load_radioml_pkl_dataset(data_path)

X, y, snr_list = [], [], []
for (mod, snr), signals in radioml_data.items():
    for complex_signal in signals:
        iq = np.vstack((complex_signal.real, complex_signal.imag))  # (2, L)
        X.append(iq)
        y.append(mod)
        snr_list.append(snr)

X = np.array(X, dtype=np.float32)    # (N, 2, 128)
y = np.array(y)
snr_list = np.array(snr_list)

print("X shape:", X.shape, "| y shape:", y.shape, "| snr shape:", snr_list.shape)
print("Unique mods:", sorted(list(set(y))))
print("Unique SNRs:", sorted(list(set(snr_list))))

# Label encoding (same pattern as your other scripts)
le = LabelEncoder()
y_enc = le.fit_transform(y).astype(np.int64)
label_names = list(le.classes_)
num_classes = len(label_names)

# 70 / 15 / 15 stratified split with fixed seed (must match training scripts)
X_train, X_tmp, y_train, y_tmp, snr_train, snr_tmp = train_test_split(
    X, y_enc, snr_list, test_size=0.30, stratify=y_enc, random_state=42
)
X_val, X_test, y_val, y_test, snr_val, snr_test = train_test_split(
    X_tmp, y_tmp, snr_tmp, test_size=0.50, stratify=y_tmp, random_state=42
)

# ========= Normalization (must match training; you're using raw IQ => "none") =========
NORM = "none"   # "none" | "zscore" | "rms"

def _norm_none(x): return x

def _norm_zscore(x, eps=1e-6):
    m = x.mean(axis=2, keepdims=True)
    s = x.std(axis=2, keepdims=True) + eps
    return (x - m) / s

def _norm_rms(x, eps=1e-8):
    rms = np.sqrt((x * x).mean(axis=2, keepdims=True)) + eps
    return x / rms

def _get_norm_fn(mode: str):
    if mode == "none":   return _norm_none
    if mode == "zscore": return _norm_zscore
    if mode == "rms":    return _norm_rms
    raise ValueError(f"Unknown normalize mode: {mode}")

def make_loaders_with_snr(
    X_tr, y_tr, snr_tr,
    X_va, y_va, snr_va,
    X_te, y_te, snr_te,
    batch_size=256,
    seed=42,
    normalize="none",
):
    g = torch.Generator(device="cpu").manual_seed(seed)
    pin = (device.type == "cuda")
    norm_fn = _get_norm_fn(normalize)

    Xt = torch.from_numpy(norm_fn(X_tr)).float()
    Xv = torch.from_numpy(norm_fn(X_va)).float()
    Xe = torch.from_numpy(norm_fn(X_te)).float()
    yt = torch.from_numpy(y_tr).long()
    yv = torch.from_numpy(y_va).long()
    ye = torch.from_numpy(y_te).long()
    st = torch.from_numpy(snr_tr).float()
    sv = torch.from_numpy(snr_va).float()
    se = torch.from_numpy(snr_te).float()

    train_loader = DataLoader(TensorDataset(Xt, yt), batch_size=batch_size,
                              shuffle=True, generator=g, pin_memory=pin)
    val_loader   = DataLoader(TensorDataset(Xv, yv), batch_size=batch_size,
                              shuffle=False, generator=g, pin_memory=pin)
    test_loader  = DataLoader(TensorDataset(Xe, ye), batch_size=batch_size,
                              shuffle=False, generator=g, pin_memory=pin)
    test_loader_snr = DataLoader(TensorDataset(Xe, ye, se), batch_size=batch_size,
                                 shuffle=False, generator=g, pin_memory=pin)
    return train_loader, val_loader, test_loader, test_loader_snr

train_loader, val_loader, test_loader, test_loader_snr = make_loaders_with_snr(
    X_train, y_train, snr_train,
    X_val,   y_val,   snr_val,
    X_test,  y_test,  snr_test,
    batch_size=256,
    seed=42,
    normalize=NORM,
)


# ========= Model definitions (copied from your code-only scripts) =========

# --- 1D CNN backbone (exactly matches 3cnn_codeonly.py) ---
def conv_block(in_channels: int,
               out_channels: int,
               kernel_size: int = 3,
               stride: int = 1,
               padding: int = 1):
    return nn.Sequential(
        nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            bias=False,
        ),
        nn.BatchNorm1d(out_channels),
        nn.ReLU(inplace=True),
    )


class AMC1DCNN(nn.Module):
    # (B,2,128) → [Conv-BN-ReLU]*2 + MaxPool(2) ×3 → GAP → 128→256→Dropout→C
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(2, 32),   conv_block(32, 32),  nn.MaxPool1d(2),  # 128→64
            conv_block(32, 64),  conv_block(64, 64),  nn.MaxPool1d(2),  # 64→32
            conv_block(64, 128), conv_block(128, 128), nn.MaxPool1d(2), # 32→16
        )
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if getattr(m, "bias", None) is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        # x: (B, 2, 128)
        x = self.features(x)          # (B, 128, 16)
        x = self.gap(x).squeeze(-1)   # (B, 128)
        return self.classifier(x)     # (B, num_classes)

class AMCRNN(nn.Module):
    """
    RNN backbone for AMC using your existing RNN design.

    Input:  (B, 2, 128)  from the loaders
    Inside: permute -> (B, 128, 2)  then GRU/LSTM
    Output: logits (B, num_classes)
    """
    def __init__(
        self,
        num_classes: int,
        cell: str = "gru",          # "gru" or "lstm"
        hidden_size: int = 256,     # we’ll override with best-arch values below
        num_layers: int = 2,
        bidirectional: bool = False,
        dropout: float = 0.3,
    ):
        super().__init__()
        input_dim = 2  # I/Q as features

        if cell.lower() == "gru":
            self.rnn = nn.GRU(
                input_size=input_dim,
                hidden_size=hidden_size,
                num_layers=num_layers,
                batch_first=True,
                bidirectional=bidirectional,
                dropout=dropout if num_layers > 1 else 0.0,
            )
        else:
            self.rnn = nn.LSTM(
                input_size=input_dim,
                hidden_size=hidden_size,
                num_layers=num_layers,
                batch_first=True,
                bidirectional=bidirectional,
                dropout=dropout if num_layers > 1 else 0.0,
            )

        out_dim = hidden_size * (2 if bidirectional else 1)
        self.fc = nn.Linear(out_dim, num_classes)

    def forward(self, x):
        # x: (B, 2, 128) → (B, 128, 2)
        x = x.transpose(1, 2)
        out, _ = self.rnn(x)       # (B, T, H*dirs)
        last = out[:, -1, :]       # (B, H*dirs)
        logits = self.fc(last)     # (B, num_classes)
        return logits


# ========= Hybrid ensemble module =========
class CNNRNNEnsemble(nn.Module):
    """
    Simple logit-level ensemble:
        logits = w_cnn * f_cnn(x) + w_rnn * f_rnn(x)
    """
    def __init__(self, cnn_model: nn.Module, rnn_model: nn.Module,
                 w_cnn: float = 0.5, w_rnn: float = 0.5):
        super().__init__()
        self.cnn = cnn_model
        self.rnn = rnn_model
        self.w_cnn = w_cnn
        self.w_rnn = w_rnn

    def forward(self, x):
        logits_cnn = self.cnn(x)
        logits_rnn = self.rnn(x)
        return self.w_cnn * logits_cnn + self.w_rnn * logits_rnn

# ========= Utility: load best manifests/checkpoints =========

def bootstrap_overall_acc(preds, targets, n_boot=10, seed=42):
    """
    Bootstrap the overall accuracy (not per-SNR).

    Returns:
      mean_acc, std_acc, acc_samples
      where acc_samples is shape (n_boot,)
    """
    rng = np.random.default_rng(seed)
    N = len(targets)
    accs = np.zeros(n_boot, dtype=np.float32)

    for b in range(n_boot):
        # sample indices with replacement
        idx = rng.integers(0, N, size=N)
        pb = preds[idx]
        tb = targets[idx]
        accs[b] = 100.0 * np.mean(pb == tb)

    return float(accs.mean()), float(accs.std()), accs


def snr_bootstrap_stats(preds, targets, snrs, n_boot=10, seed=42):
    rng = np.random.default_rng(seed)
    uniq_snrs = np.sort(np.unique(snrs))
    N = len(targets)

    # shape: (n_boot, n_snr)
    acc_boot = np.zeros((n_boot, len(uniq_snrs)), dtype=np.float32)

    for b in range(n_boot):
        # sample indices with replacement
        idx = rng.integers(0, N, size=N)
        pb = preds[idx]
        tb = targets[idx]
        sb = snrs[idx]

        for i, s in enumerate(uniq_snrs):
            mask = (sb == s)
            if np.any(mask):
                acc_boot[b, i] = 100.0 * np.mean(pb[mask] == tb[mask])
            else:
                acc_boot[b, i] = np.nan  # should not really happen

    mean_acc = np.nanmean(acc_boot, axis=0)
    std_acc  = np.nanstd(acc_boot, axis=0)
    return uniq_snrs, mean_acc, std_acc


@torch.no_grad()
def collect_preds_and_snrs(model: nn.Module, loader_snr: DataLoader):
    model.eval()
    preds, targets, snrs = [], [], []
    for xb, yb, sb in loader_snr:
        xb = xb.to(device)
        yb = yb.to(device)
        logits = model(xb)
        preds.extend(logits.argmax(1).cpu().numpy())
        targets.extend(yb.cpu().numpy())
        snrs.extend(sb.numpy())
    return np.array(preds), np.array(targets), np.array(snrs)


def load_best_cnn_from_root(cnn_root: str):
    """
    Find the best CNN checkpoint under a run root.

    Assumes:
      - best_model/ contains the final best .pth
      - FINAL_*_summary.json exists at the root (optional, for metadata)
    """
    best_model_dir = os.path.join(cnn_root, "best_model")
    ckpt_candidates = glob.glob(os.path.join(best_model_dir, "*.pth"))

    # Fallback: look in models/ if best_model is empty for some reason
    if not ckpt_candidates:
        ckpt_candidates = glob.glob(os.path.join(cnn_root, "models", "*.pth"))

    if not ckpt_candidates:
        raise FileNotFoundError(f"No CNN .pth checkpoints found under {cnn_root}")

    # Just take the first one in sorted order (there is usually only 1)
    ckpt_path = sorted(ckpt_candidates)[0]

    # Optional: try to read a FINAL_*_summary.json for nice metadata
    summary_candidates = glob.glob(os.path.join(cnn_root, "FINAL_*_summary.json"))
    name = "best_cnn"
    summary_path = None
    if summary_candidates:
        summary_path = sorted(summary_candidates)[0]
        try:
            with open(summary_path, "r") as f:
                meta = json.load(f)
            name = meta.get("name", name)
        except Exception:
            pass

    manifest = {
        "name": name,
        "ckpt_path": ckpt_path,
        "summary_json": summary_path,
        "root": cnn_root,
    }

    print(f"[CNN] Using checkpoint: {ckpt_path}")
    if summary_path:
        print(f"[CNN] Summary JSON: {summary_path}")
    return manifest



def find_best_rnn_manifest(rnn_root: str):
    """
    RNN manifests: <rnn_root>/<cfg_name>/metrics/manifest.json
    """
    manifests = []
    for dirpath, _, filenames in os.walk(rnn_root):
        if "manifest.json" in filenames:
            mpath = os.path.join(dirpath, "manifest.json")
            with open(mpath, "r") as f:
                m = json.load(f)
            manifests.append(m)
    if not manifests:
        raise FileNotFoundError(f"No RNN manifests found under {rnn_root}")
    best_m, best_val = None, -1.0
    for m in manifests:
        val = float(m.get("val_best_acc", m.get("test_acc", -1.0)))
        if val > best_val:
            best_val = val
            best_m = m
    print(f"[RNN] Best config: {best_m['name']} (val_best_acc={best_val:.2f}%)")
    return best_m

best_cnn_manifest = load_best_cnn_from_root(CNN_RUN_ROOT)
best_rnn_manifest = find_best_rnn_manifest(RNN_RUN_ROOT)

# ========= Rebuild and load the models =========
# CNN
cnn_ckpt_path = best_cnn_manifest["ckpt_path"]
cnn = AMC1DCNN(num_classes=num_classes).to(device)
cnn.load_state_dict(torch.load(cnn_ckpt_path, map_location=device))
cnn.eval()
print("Loaded best CNN checkpoint from:", cnn_ckpt_path)

# RNN (hard-coded best architecture from your ablations)
rnn_ckpt_path = best_rnn_manifest["ckpt_path"]

rnn = AMCRNN(
    num_classes=num_classes,
    cell="gru",
    hidden_size=256,
    num_layers=2,
    bidirectional=False,
    dropout=0.3,
).to(device)

rnn.load_state_dict(torch.load(rnn_ckpt_path, map_location=device))
rnn.eval()
print("Loaded best RNN checkpoint from:", rnn_ckpt_path)


# ========= Evaluation helpers =========
@torch.no_grad()
def eval_model(model: nn.Module, loader: DataLoader):
    model.eval()
    correct, total = 0, 0
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        logits = model(xb)
        preds = logits.argmax(1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)
    return 100.0 * correct / max(1, total)

@torch.no_grad()
def eval_snr_curve(model: nn.Module, loader_snr: DataLoader):
    model.eval()
    preds, targets, snrs = [], [], []
    for xb, yb, sb in loader_snr:
        xb = xb.to(device)
        yb = yb.to(device)
        logits = model(xb)
        preds.extend(logits.argmax(1).cpu().numpy())
        targets.extend(yb.cpu().numpy())
        snrs.extend(sb.numpy())
    preds   = np.array(preds)
    targets = np.array(targets)
    snrs    = np.array(snrs)

    uniq_snrs = sorted(list(set(snrs)))
    accs = []
    for s in uniq_snrs:
        idx = (snrs == s)
        accs.append(100.0 * np.mean(preds[idx] == targets[idx]))
    overall = 100.0 * np.mean(preds == targets)
    return uniq_snrs, accs, overall

# ========= Run evaluation for CNN and RNN alone =========
print("\n=== Evaluating CNN and RNN individually on the shared test set ===")
cnn_test_acc = eval_model(cnn, test_loader)
rnn_test_acc = eval_model(rnn, test_loader)

print(f"Test Acc CNN : {cnn_test_acc:.2f}%")
print(f"Test Acc RNN : {rnn_test_acc:.2f}%")

# ========= Sweep alpha for the hybrid on the *validation* set =========
# Hybrid logits: z_hyb = alpha * z_cnn + (1 - alpha) * z_rnn
alphas = [0.25, 0.5, 0.75]
alpha_results = []

print("\n=== Sweeping hybrid weights alpha on the validation set ===")
for a in alphas:
    hybrid_tmp = CNNRNNEnsemble(cnn, rnn, w_cnn=a, w_rnn=1.0 - a).to(device)
    val_acc = eval_model(hybrid_tmp, val_loader)
    alpha_results.append((a, val_acc))
    print(f"alpha={a:.2f} → Val Acc Hybrid: {val_acc:.2f}%")

# Pick the best alpha by validation accuracy
best_alpha, best_val_acc = max(alpha_results, key=lambda x: x[1])
print(f"\nBest alpha on val: {best_alpha:.2f} (val acc={best_val_acc:.2f}%)")

# ========= Final hybrid with best alpha, evaluated on test set =========
hybrid = CNNRNNEnsemble(cnn, rnn, w_cnn=best_alpha, w_rnn=1.0 - best_alpha).to(device)
hybrid.eval()

# ---- SAVE HYBRID CHECKPOINT HERE ----
hyb_ckpt_path = os.path.join(
    HYBRID_ROOT,
    f"hybrid_alpha_{best_alpha:.2f}.pth"
)

torch.save(
    {
        "hybrid_state_dict": hybrid.state_dict(),
        "w_cnn": float(best_alpha),
        "w_rnn": float(1.0 - best_alpha),
        "cnn_ckpt_path": cnn_ckpt_path,
        "rnn_ckpt_path": rnn_ckpt_path,
        "cnn_arch": "AMC1DCNN",
        "rnn_arch": "AMCRNN",
    },
    hyb_ckpt_path,
)
print("Saved hybrid checkpoint to:", hyb_ckpt_path)

hyb_test_acc = eval_model(hybrid, test_loader)
print(f"\nTest Acc Hybrid (alpha={best_alpha:.2f}): {hyb_test_acc:.2f}%")

# ========= Collect per-example predictions for each model (for bootstrap) =========
cnn_preds, cnn_targets, cnn_snrs_all = collect_preds_and_snrs(cnn,    test_loader_snr)
rnn_preds, rnn_targets, rnn_snrs_all = collect_preds_and_snrs(rnn,    test_loader_snr)
hyb_preds, hyb_targets, hyb_snrs_all = collect_preds_and_snrs(hybrid, test_loader_snr)

n_boot = 100  # number of bootstrap resamples

# ========= Overall bootstrap mean/std for each model =========
cnn_mean_acc, cnn_std_acc, cnn_boot = bootstrap_overall_acc(
    cnn_preds, cnn_targets, n_boot=n_boot, seed=0
)
rnn_mean_acc, rnn_std_acc, rnn_boot = bootstrap_overall_acc(
    rnn_preds, rnn_targets, n_boot=n_boot, seed=1
)
hyb_mean_acc, hyb_std_acc, hyb_boot = bootstrap_overall_acc(
    hyb_preds, hyb_targets, n_boot=n_boot, seed=2
)

print(f"\nBootstrap overall accuracy (n={n_boot} resamples):")
print(f"  CNN    : mean = {cnn_mean_acc:.2f}%, std = {cnn_std_acc:.2f}%")
print(f"  RNN    : mean = {rnn_mean_acc:.2f}%, std = {rnn_std_acc:.2f}%")
print(f"  Hybrid : mean = {hyb_mean_acc:.2f}%, std = {hyb_std_acc:.2f}%")

plt.figure(figsize=(6, 4))
models = ["CNN", "RNN", "Hybrid"]
means  = [cnn_mean_acc, rnn_mean_acc, hyb_mean_acc]
stds   = [cnn_std_acc,  rnn_std_acc,  hyb_std_acc]

x = np.arange(len(models))
plt.bar(x, means, yerr=stds, capsize=5)
plt.xticks(x, models)
plt.ylabel("Accuracy (%)")
plt.title(f"Bootstrap overall accuracy (mean ± std, n={n_boot})")

for i, (m, s) in enumerate(zip(means, stds)):
    plt.text(i, m + s + 0.2, f"{m:.2f}±{s:.2f}", ha="center", va="bottom", fontsize=8)

overall_bootstrap_path = os.path.join(HYBRID_ROOT, "bootstrap_overall_acc_bar.png")
plt.tight_layout()
plt.savefig(overall_bootstrap_path, dpi=300, bbox_inches="tight")
plt.close()
print("Saved:", overall_bootstrap_path)


# ========= Save overall bootstrap results as a CSV table =========
overall_table_path = os.path.join(HYBRID_ROOT, "bootstrap_overall_acc_table.csv")
with open(overall_table_path, "w") as f:
    f.write("Model,MeanAccuracyPct,StdAccuracyPct,n_boot\n")
    f.write(f"CNN,{cnn_mean_acc:.4f},{cnn_std_acc:.4f},{n_boot}\n")
    f.write(f"RNN,{rnn_mean_acc:.4f},{rnn_std_acc:.4f},{n_boot}\n")
    f.write(f"Hybrid,{hyb_mean_acc:.4f},{hyb_std_acc:.4f},{n_boot}\n")
print("Saved:", overall_table_path)



cnn_snrs, cnn_mean, cnn_std = snr_bootstrap_stats(cnn_preds, cnn_targets, cnn_snrs_all, n_boot=n_boot, seed=0)
rnn_snrs, rnn_mean, rnn_std = snr_bootstrap_stats(rnn_preds, rnn_targets, rnn_snrs_all, n_boot=n_boot, seed=1)
hyb_snrs, hyb_mean, hyb_std = snr_bootstrap_stats(hyb_preds, hyb_targets, hyb_snrs_all, n_boot=n_boot, seed=2)

# ========= Matrix-style table of bootstrap mean accuracies =========
# We assume cnn_snrs == rnn_snrs == hyb_snrs (same test set / SNRs)
snr_labels = [str(int(s)) for s in cnn_snrs]  # e.g., "-20", "-18", ..., "18"
row_labels = ["CNN", "RNN", "Hybrid"]

# Stack mean accuracies into a matrix: shape (3, n_snrs)
acc_mean_matrix = np.vstack([cnn_mean, rnn_mean, hyb_mean])

fig, ax = plt.subplots(
    figsize=(1.2 * len(snr_labels), 3.5)
)

im = ax.imshow(acc_mean_matrix, aspect="auto")

# Axis ticks / labels
ax.set_xticks(np.arange(len(snr_labels)))
ax.set_yticks(np.arange(len(row_labels)))
ax.set_xticklabels(snr_labels)
ax.set_yticklabels(row_labels)

ax.set_xlabel("SNR (dB)")
ax.set_title(f"Bootstrap Mean Accuracy (%) per SNR (n={n_boot})")

# Rotate x labels for readability
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# Add colorbar
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Accuracy (%)")

# Annotate each cell with the value
for i in range(acc_mean_matrix.shape[0]):
    for j in range(acc_mean_matrix.shape[1]):
        value = acc_mean_matrix[i, j]
        ax.text(
            j,
            i,
            f"{value:.1f}",
            ha="center",
            va="center",
            color="white" if value < 50 else "black",
            fontsize=8,
        )

matrix_path = os.path.join(HYBRID_ROOT, "bootstrap_accuracy_matrix.png")
plt.tight_layout()
plt.savefig(matrix_path, dpi=300, bbox_inches="tight")
plt.close()
print("Saved:", matrix_path)

cnn_snrs, cnn_snr_accs, _ = eval_snr_curve(cnn, test_loader_snr)
rnn_snrs, rnn_snr_accs, _ = eval_snr_curve(rnn, test_loader_snr)
hyb_snrs, hyb_snr_accs, _ = eval_snr_curve(hybrid, test_loader_snr)


# ========= Plots =========
plt.figure(figsize=(10, 5))

# CNN
plt.plot(cnn_snrs, cnn_mean, marker="o", label="CNN")
plt.fill_between(cnn_snrs, cnn_mean - cnn_std, cnn_mean + cnn_std, alpha=0.2)

# RNN
plt.plot(rnn_snrs, rnn_mean, marker="s", label="RNN")
plt.fill_between(rnn_snrs, rnn_mean - rnn_std, rnn_mean + rnn_std, alpha=0.2)

# Hybrid
plt.plot(hyb_snrs, hyb_mean, marker="^", label=f"Hybrid ({best_alpha:.2f}*CNN + {1.0-best_alpha:.2f}*RNN)")
plt.fill_between(hyb_snrs, hyb_mean - hyb_std, hyb_mean + hyb_std, alpha=0.2)

plt.xlabel("SNR (dB)")
plt.ylabel("Accuracy (%)")
plt.title(f"Accuracy vs SNR (bootstrap mean ± std, n={n_boot})")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()

snr_bootstrap_path = os.path.join(HYBRID_ROOT, "snr_curves_hybrid_bootstrap.png")
plt.savefig(snr_bootstrap_path, dpi=300, bbox_inches="tight")
plt.close()
print("Saved:", snr_bootstrap_path)


# SNR vs accuracy for all three
plt.figure(figsize=(10, 5))
plt.plot(cnn_snrs, cnn_snr_accs, marker="o", label="CNN")
plt.plot(rnn_snrs, rnn_snr_accs, marker="s", label="RNN")
plt.plot(
    hyb_snrs,
    hyb_snr_accs,
    marker="^",
    label=f"Hybrid ({best_alpha:.2f}*CNN + {1.0 - best_alpha:.2f}*RNN)",
)

plt.xlabel("SNR (dB)")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy vs SNR: CNN vs RNN vs Hybrid")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
snr_plot_path = os.path.join(HYBRID_ROOT, "snr_curves_hybrid.png")
plt.savefig(snr_plot_path, dpi=300, bbox_inches="tight")
plt.close()
print("Saved:", snr_plot_path)

# Bar chart of test accuracies
plt.figure(figsize=(6, 4))
names = ["CNN", "RNN", "Hybrid"]
vals  = [cnn_test_acc, rnn_test_acc, hyb_test_acc]
plt.bar(names, vals)
plt.ylabel("Test Accuracy (%)")
plt.title("Test Accuracy: CNN vs RNN vs Hybrid")
for i, v in enumerate(vals):
    plt.text(i, v + 0.3, f"{v:.1f}%", ha="center", va="bottom")
acc_bar_path = os.path.join(HYBRID_ROOT, "test_acc_bar_hybrid.png")
plt.savefig(acc_bar_path, dpi=300, bbox_inches="tight")
plt.close()
print("Saved:", acc_bar_path)

# ========= Save manifests for reproducibility =========
hybrid_manifest = {
    "hybrid_tag": HYBRID_TAG,
    "cnn_manifest": best_cnn_manifest,
    "rnn_manifest": best_rnn_manifest,
    "cnn_test_acc": float(cnn_test_acc),
    "rnn_test_acc": float(rnn_test_acc),
    "hybrid_test_acc": float(hyb_test_acc),
    "best_alpha": float(best_alpha),
    "alpha_candidates": [float(a) for a, _ in alpha_results],
    "alpha_val_accs": [float(acc) for _, acc in alpha_results],
    "hybrid_ckpt_path": hyb_ckpt_path,
    "snr_plot": snr_plot_path,
    "acc_bar_plot": acc_bar_path,
    "normalize": NORM,
}


with open(os.path.join(HYBRID_ROOT, "hybrid_manifest.json"), "w") as f:
    json.dump(hybrid_manifest, f, indent=2)

with open(os.path.join(HYBRID_ROOT, "cnn_rnn_best_manifests.json"), "w") as f:
    json.dump(
        {"cnn_best": best_cnn_manifest, "rnn_best": best_rnn_manifest},
        f,
        indent=2,
    )

print("\n=== DONE ===")
print("Hybrid results saved under:", HYBRID_ROOT)


Python: 3.12.12  |  Platform: Linux-6.6.105+-x86_64-with-glibc2.35
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Hybrid artifacts will go to: /content/drive/MyDrive/amc_runs/hybrid_20251202-024505
Using device: cuda
Loading RadioML 2016.10A from: /content/drive/MyDrive/AMC_datasets/RadioML2016_10A_dataset.pkl
Loading RadioML 2016.10A dataset from: /content/drive/MyDrive/AMC_datasets/RadioML2016_10A_dataset.pkl
Found 11 raw modulation types.
Loaded dataset contains 11 modulation classes and 20 SNR values.
Dataset summary: 11 mods, 20 SNRs, 220,000 total samples.
X shape: (220000, 2, 128) | y shape: (220000,) | snr shape: (220000,)
Unique mods: [np.str_('8PSK'), np.str_('AM-DSB'), np.str_('AM-SSB'), np.str_('BPSK'), np.str_('CPFSK'), np.str_('GFSK'), np.str_('PAM4'), np.str_('QAM16'), np.str_('QAM64'), np.str_('QPSK'), np.str_('WBFM')]
Unique SNRs: [np.int64(-20), np.int64(-18), np.int64(-16), np.int64(-14